# Allocating limited units across uncertain retail activity

**An independent historical decision study by Abdulaziz Aldoseri.**

At the beginning of a week, a hypothetical online-retail planner has a limited number of product units to make available. Giving more units to one product may leave fewer for another. The planner must choose the allocation before observing that week's product mix.

This notebook uses **2009–2011 UK-customer invoice activity from an unnamed retailer** to examine that trade-off. It runs the allocation model for a selected week and compares the outcome with simple policies. The target is **gross positive invoice activity**, including some lines later cancelled: it is not verified fulfilment, latent demand, measured stockouts or current retail conditions.

The decision variables are nonnegative integer allocations **qᵢ** for 30 training-selected products. The weekly total must not exceed a hypothetical unit budget **B**; optional protected minima require **qᵢ ≥ mᵢ**. Budget, preference and protection are scenario inputs, not quantities the optimization estimates. All policies receive the same budget and protection.

The objective minimizes expected weighted shortfall plus leftover units over completed past weekly observations:

`mean_s Σ_i [λ × max(d_is − q_i, 0) + (1 − λ) × max(q_i − d_is, 0)]`

Here **λ** is the priority placed on avoiding shortfall. Values 0.5, 0.8 and 0.95 correspond to shortfall:leftover weights 1:1, 4:1 and 19:1. They are preferences, not estimated costs or service targets. Units are treated as consuming equal normalized capacity; no prices, margins, physical dimensions or lead times are inferred. There is no carry-over inventory.

**Data attribution:** Chen, D. (2012). [Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) [Dataset]. UCI Machine Learning Repository. [DOI 10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes: sheet-overlap precedence, UK positive merchandise-proxy filters, training-only cohort selection and weekly aggregation. No endorsement is implied.


## 1. Open the reproduction files

**Local:** place this notebook beside `retail_model.py`, `data/` and `outputs/`, and use a Python notebook kernel with the pinned dependencies in `requirements.txt`. The local setup cell does not install packages or access the network.

**Optional Google Colab:** upload this notebook, run the next two cells and select the study's reproduction ZIP when prompted. The setup checks paths, sizes and every file against the archive manifest before executing project code. It then installs the four pinned packages. It does not mount Google Drive. A manifest verifies integrity, not publisher identity; obtain the archive from the study download link and compare its published archive hash when available. Hosted Colab execution has not been verified in this release.

The archive contains aggregate product/week data, model code and evidence. It must not contain raw workbooks or customer-level records. Existing nonempty extraction directories are not overwritten. Original notebook bytes are verified at extraction. Later setup checks allow saved outputs and parameter edits in the working notebook while still verifying model code, data and requirements.


In [ ]:
"""Standard-library ZIP validation for the retail reproduction notebook.

Checksums detect corruption and manifest inconsistency, not publisher identity.
Obtain the archive from the study's own download link and compare its published
archive checksum when available. No archive code is executed by this helper.
"""
from __future__ import annotations

from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
from zipfile import ZipFile

MAX_ARCHIVE_BYTES = 100 * 1024 * 1024
MAX_TOTAL_BYTES = 200 * 1024 * 1024
MAX_FILE_BYTES = 32 * 1024 * 1024
MAX_FILES = 2000
MANIFEST_NAME = "package_manifest.json"


def safe_name(name: str) -> str:
    if not isinstance(name, str) or not name or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError("Invalid archive path")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in name.split("/")):
        raise ValueError("Archive path must be relative without traversal")
    if any(part.casefold() in {"private", ".git", ".env"} for part in path.parts):
        raise ValueError("Private or repository-internal material is not allowed in the public package")
    if path.suffix.casefold() in {".xlsx", ".xls"}:
        raise ValueError("The public package must not contain raw source workbooks")
    return path.as_posix()


def manifest_entries(raw: bytes) -> dict:
    if len(raw) > 1024 * 1024:
        raise ValueError("Manifest exceeds size limit")
    value = json.loads(raw.decode("utf-8"))
    if not isinstance(value, dict) or not isinstance(value.get("files"), list):
        raise ValueError("Manifest must contain a files array")
    if not 1 <= len(value["files"]) <= MAX_FILES:
        raise ValueError("Invalid manifest file count")
    records, folded = {}, set()
    for item in value["files"]:
        if not isinstance(item, dict):
            raise ValueError("Invalid manifest record")
        name = safe_name(item.get("path"))
        size, digest = item.get("bytes"), item.get("sha256")
        if name == MANIFEST_NAME or name.casefold() in folded:
            raise ValueError("Duplicate, case-colliding or self-referencing manifest path")
        if isinstance(size, bool) or not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError("Invalid manifest size")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Invalid SHA-256 digest")
        records[name] = {"bytes": size, "sha256": digest}
        folded.add(name.casefold())
    if sum(record["bytes"] for record in records.values()) > MAX_TOTAL_BYTES:
        raise ValueError("Manifest total exceeds size limit")
    return records


def verify_package(directory: str | Path, allow_notebook_edits: bool = False) -> dict:
    """Verify declared files; optional working-notebook edits do not exempt code/data.

    Extraction always uses strict verification. At notebook runtime, Jupyter
    saves outputs and edited parameters into .ipynb files, so those working
    documents may differ while all executable modules/data/requirements remain
    hash-checked. Notebook paths still must exist and be regular bounded files.
    """
    root = Path(directory).resolve()
    manifest = root / MANIFEST_NAME
    if manifest.is_symlink() or not manifest.is_file():
        raise ValueError("Package manifest is missing or is a symlink")
    entries = manifest_entries(manifest.read_bytes())
    for name, item in entries.items():
        target = root / name
        if any(part.is_symlink() for part in [target, *target.parents] if part != root.parent):
            raise ValueError("Symlinks are not permitted in a package")
        if not target.resolve().is_relative_to(root) or not target.is_file():
            raise ValueError("Missing or unsafe package file")
        if allow_notebook_edits and target.suffix.casefold() == ".ipynb":
            if target.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Working notebook exceeds size limit")
            continue
        if target.stat().st_size != item["bytes"] or sha256(target.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Package checksum or size mismatch: " + name)
    return entries


def safe_extract_package(archive: bytes | str | Path, destination: str | Path) -> Path:
    """Verify all members before writing to a new/empty destination directory."""
    if isinstance(archive, bytes):
        raw = archive
    else:
        path = Path(archive)
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            raise ValueError("Archive exceeds compressed size limit")
        raw = path.read_bytes()
    if len(raw) > MAX_ARCHIVE_BYTES:
        raise ValueError("Archive exceeds compressed size limit")
    output = Path(destination)
    if output.is_symlink():
        raise ValueError("Destination must not be a symlink")
    output = output.resolve()
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise ValueError("Choose a new or empty extraction directory")
    with ZipFile(BytesIO(raw)) as archive_zip:
        infos = archive_zip.infolist()
        if len(infos) > MAX_FILES + 1:
            raise ValueError("Archive has too many entries")
        names, folded, total = {}, set(), 0
        for info in infos:
            name = safe_name(info.filename.rstrip("/") if info.is_dir() else info.filename)
            if name.casefold() in folded:
                raise ValueError("Duplicate or case-colliding archive member")
            folded.add(name.casefold())
            kind = stat.S_IFMT(info.external_attr >> 16)
            if kind not in {0, stat.S_IFREG, stat.S_IFDIR} or (kind == stat.S_IFDIR and not info.is_dir()):
                raise ValueError("Archive contains a nonregular entry")
            if info.flag_bits & 1:
                raise ValueError("Encrypted archive members are not supported")
            if info.is_dir():
                continue
            if not 0 <= info.file_size <= MAX_FILE_BYTES:
                raise ValueError("Archive member exceeds size limit")
            total += info.file_size
            names[name] = info
        if total > MAX_TOTAL_BYTES:
            raise ValueError("Archive exceeds expanded size limit")
        file_names = {name.casefold() for name in names}
        prefixes = {}
        for name in names:
            parts = PurePosixPath(name).parts
            for end in range(1, len(parts) + 1):
                prefix = "/".join(parts[:end])
                folded_prefix = prefix.casefold()
                if folded_prefix in prefixes and prefixes[folded_prefix] != prefix:
                    raise ValueError("Inconsistent case in archive path components")
                prefixes[folded_prefix] = prefix
                if end < len(parts) and folded_prefix in file_names:
                    raise ValueError("Archive file conflicts with a required directory")
        if MANIFEST_NAME not in names:
            raise ValueError("Archive root must contain package_manifest.json")
        if names[MANIFEST_NAME].file_size > 1024 * 1024:
            raise ValueError("Manifest exceeds size limit")
        manifest = archive_zip.read(names[MANIFEST_NAME])
        entries = manifest_entries(manifest)
        if set(names) != set(entries) | {MANIFEST_NAME}:
            raise ValueError("Every archive file must appear exactly once in the manifest")
        verified = {MANIFEST_NAME: manifest}
        for name, item in entries.items():
            info = names[name]
            if info.file_size != item["bytes"]:
                raise ValueError("Member size disagrees with manifest")
            with archive_zip.open(info) as member:
                contents = member.read(MAX_FILE_BYTES + 1)
            if len(contents) != item["bytes"] or sha256(contents).hexdigest() != item["sha256"]:
                raise ValueError("Member checksum disagrees with manifest: " + name)
            verified[name] = contents
    # The manifest is validated before any data or executable source is written.
    output.mkdir(parents=True, exist_ok=True)
    for name, contents in verified.items():
        target = output / name
        if not target.resolve().is_relative_to(output):
            raise ValueError("Unsafe extraction target")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as handle:
            handle.write(contents)
    verify_package(output)
    return output


In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False

PROJECT = Path.cwd().resolve()
if not (PROJECT / "retail_model.py").is_file():
    extracted = PROJECT / "retail-allocation-reproduction"
    if (extracted / "retail_model.py").is_file():
        verify_package(extracted, allow_notebook_edits=True)
        PROJECT = extracted
    elif IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one reproduction ZIP.")
        filename, contents = next(iter(uploaded.items()))
        if not filename.lower().endswith(".zip"):
            raise ValueError("Choose the study reproduction ZIP.")
        PROJECT = safe_extract_package(contents, extracted)
    else:
        raise FileNotFoundError("Run the notebook from the extracted project folder containing retail_model.py.")

if (PROJECT / "package_manifest.json").is_file():
    verify_package(PROJECT, allow_notebook_edits=True)
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(PROJECT / "requirements.txt")])

sys.path.insert(0, str(PROJECT))
import pandas as pd
from retail_model import all_policies, budget_from_history, evaluate
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

index = json.loads((PROJECT / "outputs/public_index.json").read_text(encoding="utf-8"))
products = index["products"]
codes = [p["code"] for p in products]
names = {p["code"]: p["name"] for p in products}
window = int(index["meta"]["selected_window"])
assert len(codes) == 30 and len(set(codes)) == 30 and codes == sorted(codes)
assert len(index["weeks"]) == 23
assert window in (8, 13, 26)
print(f"Loaded {len(codes)} fixed products and 23 held-out weeks. Frozen scenario window: {window} weeks.")


## 2. What information can the planner use?

The product cohort, names and extreme-order caps were fixed using 55 complete training weeks. Validation used 26 later weeks to select one global scenario window from 8, 13 and 26. The selected window is then frozen for all 23 test weeks. At each test Monday, only completed preceding weeks enter the budget or empirical scenarios. The actual week is used for evaluation and the explicitly unattainable hindsight reference.

The budget is a factor times the sum of trailing-eight-week means, rounded half-up to an integer. It is fixed independently of the policy, selected scenario window, priority and protection.

**Comparison policies:**

- **Scenario allocation:** exact integer allocation minimizing expected weighted mismatch using the frozen empirical history window. Ties minimize allocated units, then follow product-code order. The additive objective depends on marginal product distributions; it does not claim a benefit from cross-product correlation.
- **Eight-week mean:** round product means; if their total exceeds the budget, ration proportionally using a deterministic largest-remainder rule.
- **Same week last year:** use recorded quantities 364 days earlier with the same rationing rule.
- **Equal allocation:** water-fill final product quantities, respecting protection and the mean-rule total. It need not spend an ample budget.
- **Perfect-information reference:** knows the actual week's activity in advance. This is a bound, not an available operating policy.

The recorded timestamps have no supplied timezone, and historical source cleaning is not an as-issued operational backtest. A zero week means zero recorded activity, not established zero customer demand. No real replenishment recommendation follows from this normalized experiment.


In [ ]:
# Validation results were generated before held-out scoring.
display(pd.DataFrame(index["validation"]).sort_values("window"))

# Confirm the displayed test records match the prepared aggregate panel.
weekly = pd.read_csv(PROJECT / "data/weekly_quantities.csv", dtype={"StockCode": str})
assert len(weekly) == 104 * 30
assert weekly.groupby("split").week_start.nunique().to_dict() == {"train": 55, "validation": 26, "test": 23}
panel = weekly.pivot(index="week_start", columns="StockCode", values="primary_units").reindex(columns=codes).sort_index()
for record in index["weeks"]:
    at = panel.index.get_loc(record["week"])
    assert record["actual"] == panel.iloc[at].astype(int).tolist()
    assert record["history"] == panel.iloc[at - 26:at].astype(int).values.tolist()
    assert record["seasonal"] == panel.iloc[at - 52].astype(int).tolist()
print("All 23 weeks, 26-week histories and seasonal observations match the aggregate CSV.")


## 3. Choose a week and a planning scenario

Change the values below and rerun the remaining cells. The default week, **4 July 2011**, was chosen by calendar before model evaluation. All 23 test weeks remain available; the last begins 28 November 2011.

`BUDGET_FACTOR` can be 0.5, 0.75, 1.0 or 1.25. `SHORTFALL_PRIORITY` can be 0.5, 0.8 or 0.95. To reserve units for a product, enter one listed product code and a nonnegative integer `MINIMUM_UNITS`. These units are part of the same budget; they are not additional capacity. A minimum exceeding the budget is infeasible and raises an error rather than silently changing the budget.


In [ ]:
WEEK = "2011-07-04"
BUDGET_FACTOR = 1.0
SHORTFALL_PRIORITY = 0.8
PROTECTED_CODE = None   # Example: "85123A"; None means no protected product.
MINIMUM_UNITS = 0       # Nonnegative integer, not a percentage.

if BUDGET_FACTOR not in index["meta"]["budget_factors"]:
    raise ValueError("Choose a predeclared budget factor: 0.5, 0.75, 1.0 or 1.25.")
if SHORTFALL_PRIORITY not in index["meta"]["priorities"]:
    raise ValueError("Choose a predeclared priority: 0.5, 0.8 or 0.95.")
if isinstance(MINIMUM_UNITS, bool) or not isinstance(MINIMUM_UNITS, int) or MINIMUM_UNITS < 0:
    raise ValueError("MINIMUM_UNITS must be a nonnegative integer.")
if PROTECTED_CODE is not None and PROTECTED_CODE not in codes:
    raise ValueError("Protected product must belong to the fixed cohort.")
if PROTECTED_CODE is None and MINIMUM_UNITS:
    raise ValueError("Select a protected product before setting a positive minimum.")
matching = [record for record in index["weeks"] if record["week"] == WEEK]
if len(matching) != 1:
    raise ValueError("Choose one of the 23 held-out Monday weeks listed below.")
record = matching[0]
history, actual, seasonal = record["history"], record["actual"], record["seasonal"]
budget = budget_from_history(history, BUDGET_FACTOR)
minima = [MINIMUM_UNITS if code == PROTECTED_CODE else 0 for code in codes]
if sum(minima) > budget:
    raise ValueError(f"Infeasible protection: {sum(minima):,} reserved units exceed the {budget:,}-unit budget.")
print("Available test Mondays:", ", ".join(record["week"] for record in index["weeks"]))
print(f"Selected week: {WEEK}; hypothetical budget: {budget:,} units; shortfall priority: {SHORTFALL_PRIORITY}.")
print("Protected requirement:", "none" if PROTECTED_CODE is None else f"{PROTECTED_CODE}: {MINIMUM_UNITS:,} units")
display(pd.DataFrame(products))


## 4. Run the allocation model

This cell computes the allocations from the supplied history; it does not merely load a precomputed selected-week answer. Expected loss uses the same historical scenarios for every policy. Realized loss evaluates the chosen allocation against that week's recorded positive invoice quantities.

**Weighted loss** is `priority × shortfall + (1 − priority) × leftover`, in weighted units. **Recorded-activity coverage** is the proportion of observed units the allocation could cover, pooled across products. It is undefined when recorded activity is zero; it is not achieved customer service or a measured fill rate. Compare scores only at the same priority.


In [ ]:
labels = {
    "scenario": "Scenario allocation",
    "mean": "Eight-week mean",
    "seasonal": "Same week last year",
    "equal": "Equal allocation",
    "hindsight": "Perfect-information reference",
}
allocations = all_policies(codes, history, seasonal, actual, budget, SHORTFALL_PRIORITY, window, minima)
metrics = []
for policy, allocation in allocations.items():
    assert all(isinstance(q, int) and q >= m for q, m in zip(allocation, minima))
    assert sum(allocation) <= budget
    metric = evaluate(allocation, actual, SHORTFALL_PRIORITY, budget, history[-window:])
    metrics.append({"policy": labels[policy], "allocated_units": sum(allocation), **metric})
display(pd.DataFrame(metrics).rename(columns={"loss": "realized_weighted_loss", "expected_loss": "expected_weighted_loss", "coverage": "recorded_activity_coverage"}))

allocation_table = pd.DataFrame({"code": codes, "product": [names[c] for c in codes], "recorded_units": actual, "protected_minimum": minima})
for policy, allocation in allocations.items():
    allocation_table[labels[policy]] = allocation
display(allocation_table)

if any(minima):
    unprotected = all_policies(codes, history, seasonal, actual, budget, SHORTFALL_PRIORITY, window)["scenario"]
    change = [a - b for a, b in zip(allocations["scenario"], unprotected)]
    changed = pd.DataFrame({"code": codes, "product": [names[c] for c in codes], "allocation_change": change})
    display(changed.loc[changed.allocation_change.ne(0)])
    print("Net additional units allocated:", sum(change), "; displaced units across losing products:", -sum(min(v, 0) for v in change))
    print("Protection can use previously unused budget; it does not necessarily displace another product.")


## 5. Read all held-out weeks

The next cell recomputes all 23 test weeks for the selected budget factor and priority with **zero protected minima**, then checks their mean scores against the packaged results. This is a small result check, not a rerun of window selection or the complete evaluation pipeline. It retains both good and adverse weeks. Any protected single-week scenario above is separate from this unprotected whole-period evidence.

The mean gives every week equal weight. Whole-period coverage pools observed units rather than averaging weekly percentages. A negative scenario-minus-baseline loss difference favours the scenario method at the selected preference; it does not establish general superiority.


In [ ]:
test_rows = []
for week_record in index["weeks"]:
    hist, observed = week_record["history"], week_record["actual"]
    B = budget_from_history(hist, BUDGET_FACTOR)
    choices = all_policies(codes, hist, week_record["seasonal"], observed, B, SHORTFALL_PRIORITY, window)
    for policy, choice in choices.items():
        values = evaluate(choice, observed, SHORTFALL_PRIORITY, B, hist[-window:])
        test_rows.append({"week": week_record["week"], "policy": policy, "observed_units": sum(observed), **values})
test = pd.DataFrame(test_rows)
summary = test.groupby("policy", sort=False).agg(
    mean_loss=("loss", "mean"), mean_shortfall=("shortfall", "mean"),
    mean_leftover=("leftover", "mean"), mean_unused=("unused", "mean"),
    observed_units=("observed_units", "sum"), shortfall_units=("shortfall", "sum"),
).reset_index()
summary["pooled_recorded_activity_coverage"] = (summary.observed_units - summary.shortfall_units) / summary.observed_units.replace(0, float("nan"))
stored = [row for row in index["summaries"] if row["factor"] == BUDGET_FACTOR and row["priority"] == SHORTFALL_PRIORITY]
for row in stored:
    computed = summary.loc[summary.policy.eq(row["policy"])].iloc[0]
    for field in ["mean_loss", "mean_shortfall", "mean_leftover", "mean_unused"]:
        assert abs(float(computed[field]) - row[field]) < 1e-8
assert len(stored) == 5
display(summary.drop(columns=["observed_units", "shortfall_units"]).assign(policy=lambda frame: frame.policy.map(labels)))

by_week = test.pivot(index="week", columns="policy", values="loss")
for baseline in ["mean", "seasonal", "equal"]:
    by_week[f"scenario_minus_{baseline}"] = by_week.scenario - by_week[baseline]
display(by_week.reset_index())

paired = [row for row in index["comparisons"] if row["factor"] == BUDGET_FACTOR and row["priority"] == SHORTFALL_PRIORITY]
display(pd.DataFrame(paired))
print("Intervals are descriptive four-week moving-block resampling sensitivities from only 23 weeks, not population guarantees.")


## 6. Check the preparation sensitivities

These packaged results retain the same cohort and frozen primary-selected scenario window. They separately collapse exact repeats, remove only uniquely matched same-week reversals, or cap positive lines at product-specific training 99th-percentile thresholds. Ambiguous or later-week cancellations remain unmatched. The cap changes the activity target; it does not identify erroneous orders. Compare policies within the same variant, budget and preference rather than treating lower scores on a smaller target as a model improvement.


In [ ]:
sensitivity = [row for row in index["sensitivities"] if row["factor"] == BUDGET_FACTOR and row["priority"] == SHORTFALL_PRIORITY]
display(pd.DataFrame(sensitivity))


## 7. Reproduction and limits

The main model uses an exact separable marginal-allocation algorithm for this integer scenario problem. Independent small-case MILP checks and adversarial tests are included in the reproduction evidence. No operational deployment or realized business impact is claimed.

The complete local pipeline runs in two phases: `python pipeline.py --phase validate`, then `python pipeline.py --phase evaluate`, using the pinned requirements. Read `PROTOCOL.json` and `METHODS.md` first; the protocol fixes chronology, window selection, rounding, ties, baselines and sensitivities. Preserve the supplied frozen result files; existing-file guards prevent silently replacing frozen choices. This notebook intentionally does not trigger the complete pipeline.

The aggregate CSVs can be regenerated from the exact official UCI workbook, obtained separately:

```bash
python prepare_data.py --workbook /path/to/online_retail_II.xlsx --output-dir /path/to/aggregate-output --public-only
```

The script checks the frozen source SHA-256, handles the two-sheet overlap, reproduces the ordered filters and training-only cohort, and writes aggregate outputs without customer-level exports. The workbook is not included in this package. Records are assumed available by week-end; missing transactions, availability restrictions and invoice adjustments cannot be reconstructed from the source.

One historical company and 23 held-out weeks provide a bounded decision experiment. There are no observed on-hand inventories, lost sales, acquisition costs, product dimensions or lead times. Selling prices are not purchase costs or margins. Neither allocation quality nor forecast accuracy establishes causal sales gains, financial savings or a current retail operating recommendation.

**Execution record:** the notebook's code cells were checked sequentially in a local Python runtime. The ZIP setup was tested locally, including traversal, checksum, size, overwrite and symlink failures. No hosted Google Colab run is claimed.
